# 02 - OWLv2 ingredient detection: tune on dev photos, then test once on new photos
1. Runtime > Change runtime type > **T4 GPU**.
2. **Dev photos** (10-20) go in `MyDrive/AaharAI/photos/`; fill `labels.csv` (names from `vocab.DETECT`, separated by `;`). Names are case-insensitive.
3. Run all cells: threshold sweep, error report (misses are split into *in vocabulary* and *not in vocabulary*), prompt-template comparison.
4. Pick a setting. Then photograph 15-20 **new** items into `MyDrive/AaharAI/photos_test/`, fill its `labels.csv`, and run the last cell **once**. Do not tune on the test photos.
Do not save this notebook back to GitHub.

In [ ]:
GITHUB_USER = "OnYourLeftover"

from google.colab import drive
drive.mount("/content/drive")
!pip -q install rapidfuzz
import sys
!git -C /content/AaharAI pull || git clone https://github.com/{GITHUB_USER}/AaharAI.git /content/AaharAI
sys.path.insert(0, "/content/AaharAI")
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from pathlib import Path
from PIL import Image
from torchvision.ops import nms
from transformers import Owlv2Processor, Owlv2ForObjectDetection
from src import vocab, detect_eval as de

dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)
assert dev == "cuda", "Switch the runtime to a T4 GPU (Runtime > Change runtime type)."

PHOTOS = Path("/content/drive/MyDrive/AaharAI/photos")
RESULTS = Path("/content/drive/MyDrive/AaharAI/results"); RESULTS.mkdir(parents=True, exist_ok=True)
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
imgs = sorted(p for p in PHOTOS.glob("*") if p.suffix.lower() in IMG_EXT)
print(len(imgs), "photos found in", PHOTOS)

name = "google/owlv2-base-patch16-ensemble"
proc = Owlv2Processor.from_pretrained(name)
model = Owlv2ForObjectDetection.from_pretrained(name).to(dev).eval()

PROMPT_FIX = {"brinjal - all varieties": "brinjal (eggplant)", "celery stalk": "celery", "spring onions": "spring onion",
              "button mushroom": "mushroom", "peas": "green peas", "french beans": "green beans",
              "yogurt": "bowl of curd (yogurt)", "capsicum": "capsicum (bell pepper)",
              "coriander leaves": "coriander leaves (cilantro)", "radish": "white radish (mooli)", "corn": "corn cob"}
NAMES = sorted(vocab.DETECT)
TEMPLATES = {"plain": "a photo of {}", "fresh": "a photo of a fresh {}", "bare": "{}"}
VOCAB = set(NAMES)
print(len(NAMES), "ingredient prompts")

In [ ]:
def prompts(template="plain"):
    return [TEMPLATES[template].format(PROMPT_FIX.get(n, n)) for n in NAMES]

def forward_all(template="plain"):
    """One forward pass per photo; thresholds are applied afterwards in plain Python (see src/detect_eval.py)."""
    cache, texts = {}, prompts(template)
    for p in imgs:
        img = Image.open(p).convert("RGB"); img.thumbnail((1600, 1600))
        inputs = proc(text=[texts], images=img, return_tensors="pt").to(dev)
        with torch.no_grad():
            cache[p.name] = (model(**inputs), max(img.size))
    return cache

def decode(cache, thr=0.1, iou=0.5):
    """Best score per label per photo, after class-agnostic NMS. thr=0.1 keeps everything we might want to sweep."""
    raw = {}
    for photo, (out, s) in cache.items():
        res = proc.post_process_object_detection(out, threshold=thr, target_sizes=torch.tensor([[s, s]]).to(dev))[0]
        best = {}
        if len(res["scores"]):
            keep = nms(res["boxes"], res["scores"], iou_threshold=iou)
            for i in keep.tolist():
                n = NAMES[int(res["labels"][i])]
                best[n] = max(best.get(n, 0.0), float(res["scores"][i]))
        raw[photo] = best
    return raw

raw = decode(forward_all("plain"))
for photo, scores in raw.items():
    print(photo, "->", {k: round(v, 2) for k, v in sorted(scores.items(), key=lambda kv: -kv[1]) if v >= 0.3})

## Labels for the dev photos
The first run creates `labels.csv` next to the photos. Fill the `ingredients` column (e.g. `onion;tomato;radish`), then run the next cells.

In [ ]:
import pandas as pd
LABELS = PHOTOS / "labels.csv"
if not LABELS.exists():
    pd.DataFrame({"image": [p.name for p in imgs], "ingredients": ""}).to_csv(LABELS, index=False)
    print("Created", LABELS, "- fill it in, then run the cells below.")
else:
    truth = de.load_truth(pd.read_csv(LABELS).fillna(""))
    print(len(truth), "labelled photos")

In [ ]:
sw = de.sweep(raw, truth, VOCAB)
print("Top settings (f1_in_vocab ignores ingredients you labelled that are not in the detector vocabulary):")
print(sw.head(8).round(2).to_string(index=False))
best = sw.iloc[0]
rep, invented, missed_in, missed_oov = de.error_report(raw, truth, VOCAB, best.thr, best.rel)
print(f"\nBest setting: threshold {best.thr}, relative {best.rel}  ->  precision {best.precision:.2f}, recall in vocabulary {best.recall_in_vocab:.2f}, recall overall {best.recall_all:.2f}")
print("most invented:", invented.most_common(8))
print("missed, in vocabulary:", missed_in.most_common(8))
print("missed, NOT in vocabulary (add to vocab.DETECT if they matter):", missed_oov.most_common(8))
rep.to_csv(RESULTS / "owlv2_per_photo.csv", index=False); sw.to_csv(RESULTS / "owlv2_sweep.csv", index=False)
print("\n", rep[["image", "truth", "false_pos", "missed_in_vocab"]].to_string(index=False))

## Prompt templates (a cheap way to improve precision)

In [ ]:
rows = []
for tname in TEMPLATES:
    r = raw if tname == "plain" else decode(forward_all(tname))
    s = de.sweep(r, truth, VOCAB).iloc[0]
    rows.append({"template": tname, "thr": s.thr, "rel": s.rel, "precision": s.precision,
                 "recall_in_vocab": s.recall_in_vocab, "f1_in_vocab": s.f1_in_vocab})
print(pd.DataFrame(rows).round(2).to_string(index=False))

## Final test (run ONCE, on photos you did not tune on)
Put 15-20 new photos in `MyDrive/AaharAI/photos_test/`, set the chosen setting below, and run.

In [ ]:
THR, REL, TEMPLATE = 0.4, 0.0, "plain"      # <-- set these from the dev results above
TEST = Path("/content/drive/MyDrive/AaharAI/photos_test")
imgs = sorted(p for p in TEST.glob("*") if p.suffix.lower() in IMG_EXT)      # forward_all reads this list
print(len(imgs), "test photos")
raw_t = decode(forward_all(TEMPLATE))
L = TEST / "labels.csv"
if not L.exists():
    pd.DataFrame({"image": [p.name for p in imgs], "ingredients": ""}).to_csv(L, index=False)
    print("Created", L, "- fill it in and run this cell again.")
else:
    truth_t = de.load_truth(pd.read_csv(L).fillna(""))
    res = de.evaluate(raw_t, truth_t, VOCAB, THR, REL)
    print({k: round(v, 2) if isinstance(v, float) else v for k, v in res.items()})
    rep_t, inv_t, miss_t, oov_t = de.error_report(raw_t, truth_t, VOCAB, THR, REL)
    print("most invented:", inv_t.most_common(8)); print("missed in vocabulary:", miss_t.most_common(8)); print("missed NOT in vocabulary:", oov_t.most_common(8))
    rep_t.to_csv(RESULTS / "owlv2_test_per_photo.csv", index=False)